## Bayesian methods of hyperparameter optimization

In addition to the random search and the grid search methods for selecting optimal hyperparameters, we can use Bayesian methods of probabilities to select the optimal hyperparameters for an algorithm.

In this case study, we will be using the BayesianOptimization library to perform hyperparameter tuning. This library has very good documentation which you can find here: https://github.com/fmfn/BayesianOptimization

You will need to install the Bayesian optimization module. Running a cell with an exclamation point in the beginning of the command will run it as a shell command — please do this to install this module from our notebook in the cell below.

In [1]:
#! pip install bayesian-optimization lightgbm catboost

In [1]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import lightgbm
from bayes_opt import BayesianOptimization
from catboost import CatBoostClassifier, cv, Pool

In [2]:
import os
os.listdir()

['.DS_Store',
 'Bayesian_optimization_case_study.ipynb',
 'Optuna_Bayesian_optimization_case_study.ipynb',
 '.github',
 'data']

## How does Bayesian optimization work?

Bayesian optimization works by constructing a posterior distribution of functions (Gaussian process) that best describes the function you want to optimize. As the number of observations grows, the posterior distribution improves, and the algorithm becomes more certain of which regions in parameter space are worth exploring and which are not, as seen in the picture below.

<img src="https://github.com/fmfn/BayesianOptimization/blob/master/examples/bo_example.png?raw=true" />
As you iterate over and over, the algorithm balances its needs of exploration and exploitation while taking into account what it knows about the target function. At each step, a Gaussian Process is fitted to the known samples (points previously explored), and the posterior distribution, combined with an exploration strategy (such as UCB — aka Upper Confidence Bound), or EI (Expected Improvement). This process is used to determine the next point that should be explored (see the gif below).
<img src="https://github.com/fmfn/BayesianOptimization/raw/master/examples/bayesian_optimization.gif" />

## Let's look at a simple example

The first step is to create an optimizer. It uses two items:
* function to optimize
* bounds of parameters

The function is the procedure that counts metrics of our model quality. The important thing is that our optimization will maximize the value on function. Smaller metrics are best. Hint: don't forget to use negative metric values.

Here we define our simple function we want to optimize.

In [3]:
def simple_func(a, b):
    return a + b

Now, we define our bounds of the parameters to optimize, within the Bayesian optimizer.

The main parameters of the BayesianOptimization function are:

1. The function to optimize (e.g., simple_func)
2. The parameter bounds (e.g., {'a': (1, 3), 'b': (4, 7)})

In [ ]:
# Initialize BayesianOptimization object

optimizer = BayesianOptimization(
    simple_func, # Add function to optimize
    {'a': (1, 3), # Define search space for parameter bounds of 'a' and 'b'
    'b': (4, 7)})

# Optimizer will use Bayesian optimization to search for the combination of 'a' and 'b' that gives the highest value of ('maximizes') simple_func(a, b)

The main arguments used when you call the maximize() method on the optimizer object:

* **n_iter:** This is how many steps of Bayesian optimization you want to perform. The more steps, the more likely you are to find a good maximum.

n_iter sets how many times the optimizer will try new parameter values after the initial random explorations. Each step, it uses what it has learned so far to pick the next best combination to test. 

* **init_points:** This is how many steps of random exploration you want to perform. Random exploration can help by diversifying the exploration space.

This is the number of initial random explorations before Bayesian optimization starts. These initial random trials help the optimizer gather information about the search space, so it can build a better model of how the parameters affect the function. 

**Let's run an example where we use the optimizer to find the best values to maximize the target value for a and b given the inputs of 3 and 2.**

In [6]:
optimizer.maximize(3,2)

|   iter    |  target   |     a     |     b     |
-------------------------------------------------
| 6         | 7.3903495 | 1.5235933 | 5.8667562 |
| 7         | 8.4491660 | 2.2604121 | 6.1887539 |
| 8         | 7.4069174 | 1.1681446 | 6.2387728 |
| 9         | 9.0673881 | 2.9273368 | 6.1400512 |
| 10        | 10.0      | 3.0       | 7.0       |


Great, now let's print the best parameters and the associated maximized target.

In [7]:
print(optimizer.max['params']);optimizer.max['target']

{'a': np.float64(3.0), 'b': np.float64(7.0)}


np.float64(10.0)

## Test it on real data using the Light GBM

The dataset we will be working with is the famous flight departures dataset. Our modeling goal will be to predict if a flight departure is going to be delayed by 15 minutes based on the other attributes in our dataset. As part of this modeling exercise, we will use Bayesian hyperparameter optimization to identify the best parameters for our model.

**<font color='teal'> You can load the zipped csv files just as you would regular csv files using Pandas read_csv. In the next cell load the train and test data into two seperate dataframes. </font>**


In [8]:
train_df = pd.read_csv('data/flight_delays_train.csv')
test_df = pd.read_csv('data/flight_delays_test.csv')

**<font color='teal'> Print the top five rows of the train dataframe and review the columns in the data. </font>**

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance,dep_delayed_15min
0,c-8,c-21,c-7,1934,AA,ATL,DFW,732,N
1,c-4,c-20,c-3,1548,US,PIT,MCO,834,N
2,c-9,c-2,c-5,1422,XE,RDU,CLE,416,N
3,c-11,c-25,c-6,1015,OO,DEN,MEM,872,N
4,c-10,c-7,c-6,1828,WN,MDW,OMA,423,Y


In [ ]:
train_df.head()

**<font color='teal'> Use the describe function to review the numeric columns in the train dataframe. </font>**

In [10]:
train_df.describe()

,DepTime,Distance
count,100000.000000,100000.00000
mean,1341.523880,729.39716
std,476.378445,574.61686
min,1.000000,30.00000
25%,931.000000,317.00000
50%,1330.000000,575.00000
75%,1733.000000,957.00000
max,2534.000000,4962.00000


Notice, `DepTime` is the departure time in a numeric representation in 2400 hours. 

 **<font color='teal'>The response variable is 'dep_delayed_15min' which is a categorical column, so we need to map the Y for yes and N for no values to 1 and 0. Run the code in the next cell to do this.</font>**

In [11]:
train_df = train_df[train_df.DepTime <= 2400].copy()
y_train = train_df['dep_delayed_15min'].map({'Y': 1, 'N': 0}).values

## Feature Engineering
Use these defined functions to create additional features for the model. Run the cell to add the functions to your workspace.

In [ ]:
# Define a function to encode categorical values as integers.
# LabelEncoder().fit_transform(df_column) fits a label encoder to the column and transforms each unique value to an integer.
# Returns the encoded column.

def label_enc(df_column):
    df_column = LabelEncoder().fit_transform(df_column)
    return df_column

In [ ]:
# Multiply the input value by 2π/period to scale it to a full cycle.
# Returns the sine of the scaled value (useful for cyclical features like time).

def make_harmonic_features_sin(value, period=2400):
    value *= 2 * np.pi / period 
    return np.sin(value)

# Same as above, but returns the cosine of the scaled value.

def make_harmonic_features_cos(value, period=2400):
    value *= 2 * np.pi / period 
    return np.cos(value)

In [ ]:

def feature_eng(df):
    # Concatenate the origin and destination airport codes.
    df['flight'] = df['Origin']+df['Dest']

    # Extract month number from string (in this case, c-##) and convert to integer.
    df['Month'] = df.Month.map(lambda x: x.split('-')[-1]).astype('int32')

    # Extract day number from string and convert to integer.
    df['DayofMonth'] = df.DayofMonth.map(lambda x: x.split('-')[-1]).astype('uint8')

    # Create binary features for whether the day is at the beginning, middle, or end of the month.
    df['begin_of_month'] = (df['DayofMonth'] < 10).astype('uint8')
    df['midddle_of_month'] = ((df['DayofMonth'] >= 10)&(df['DayofMonth'] < 20)).astype('uint8')
    df['end_of_month'] = (df['DayofMonth'] >= 20).astype('uint8')

    # Extract day of week number from string and convert to integer.
    df['DayOfWeek'] = df.DayOfWeek.map(lambda x: x.split('-')[-1]).astype('uint8')

    # Convert departure time (e.g., 1300) to hour (e.g., 13).
    df['hour'] = df.DepTime.map(lambda x: x/100).astype('int32')

    # Create binary features for different times of day.
    df['morning'] = df['hour'].map(lambda x: 1 if (x <= 11)& (x >= 7) else 0).astype('uint8')
    df['day'] = df['hour'].map(lambda x: 1 if (x >= 12) & (x <= 18) else 0).astype('uint8')
    df['evening'] = df['hour'].map(lambda x: 1 if (x >= 19) & (x <= 23) else 0).astype('uint8')
    df['night'] = df['hour'].map(lambda x: 1 if (x >= 0) & (x <= 6) else 0).astype('int32')

    # Create binary features for each season based on month.
    df['winter'] = df['Month'].map(lambda x: x in [12, 1, 2]).astype('int32')
    df['spring'] = df['Month'].map(lambda x: x in [3, 4, 5]).astype('int32')
    df['summer'] = df['Month'].map(lambda x: x in [6, 7, 8]).astype('int32')
    df['autumn'] = df['Month'].map(lambda x: x in [9, 10, 11]).astype('int32')

    # Create binary features for holidays (Friday (5), Saturday (6), Sunday (7)) and weekdays.
    df['holiday'] = (df['DayOfWeek'] >= 5).astype(int) 
    df['weekday'] = (df['DayOfWeek'] < 5).astype(int)

    # Add count-based features: number of flights per destination/origin per month, total flights per destination/origin, and flights per carrier (overall and per month).
    df['airport_dest_per_month'] = df.groupby(['Dest', 'Month'])['Dest'].transform('count')
    df['airport_origin_per_month'] = df.groupby(['Origin', 'Month'])['Origin'].transform('count')
    df['airport_dest_count'] = df.groupby(['Dest'])['Dest'].transform('count')
    df['airport_origin_count'] = df.groupby(['Origin'])['Origin'].transform('count')
    df['carrier_count'] = df.groupby(['UniqueCarrier'])['Dest'].transform('count')
    df['carrier_count_per month'] = df.groupby(['UniqueCarrier', 'Month'])['Dest'].transform('count')

    # Create cyclical (harmonic) features for departure time.
    df['deptime_cos'] = df['DepTime'].map(make_harmonic_features_cos)
    df['deptime_sin'] = df['DepTime'].map(make_harmonic_features_sin)

    # Create combined features by concatenation of flight, destination, origin, and carrier.
    df['flightUC'] = df['flight']+df['UniqueCarrier']
    df['DestUC'] = df['Dest']+df['UniqueCarrier']
    df['OriginUC'] = df['Origin']+df['UniqueCarrier']

    # Drop the original 'DepTime' column because its information is now captured in new features. Keeping it would be redundant and could lead to multicollinearity.
    return df.drop('DepTime', axis=1)

Concatenate by stacking (the default) the training and testing dataframes and then apply the earlier defined feature engineering functions to the full dataframe by calling the feature_eng() function.

In [13]:
full_df = pd.concat([train_df.drop('dep_delayed_15min', axis=1), test_df])
full_df = feature_eng(full_df)

In [14]:
full_df.head()

,Month,DayofMonth,DayOfWeek,UniqueCarrier,Origin,Dest,Distance,flight,begin_of_month,midddle_of_month,...,airport_origin_per_month,airport_dest_count,airport_origin_count,carrier_count,carrier_count_per month,deptime_cos,deptime_sin,flightUC,DestUC,OriginUC
0,8,21,7,AA,ATL,DFW,732,ATLDFW,0,0,...,1016,8290,11375,18024,1569,0.343660,-0.939094,ATLDFWAA,DFWAA,ATLAA
1,4,20,3,US,PIT,MCO,834,PITMCO,0,0,...,105,3523,1390,13069,1094,-0.612907,-0.790155,PITMCOUS,MCOUS,PITUS
2,9,2,5,XE,RDU,CLE,416,RDUCLE,1,0,...,136,2246,1747,11737,977,-0.835807,-0.549023,RDUCLEXE,CLEXE,RDUXE
3,11,25,6,OO,DEN,MEM,872,DENMEM,0,0,...,514,1785,6222,15343,1242,-0.884988,0.465615,DENMEMOO,MEMOO,DENOO
4,10,7,6,WN,MDW,OMA,423,MDWOMA,1,0,...,226,687,2571,30958,2674,0.073238,-0.997314,MDWOMAWN,OMAWN,MDWWN


In [ ]:
# This code block applies the label_enc function to categorical columns in the full_df DataFrame. 'label_enc' converts categorical string values into integer labels, which are easier for machine learning models to process.

for column in ['UniqueCarrier', 'Origin', 'Dest','flight',  'flightUC', 'DestUC', 'OriginUC']:
    full_df[column] = label_enc(full_df[column])

In [18]:
full_df

,Month,DayofMonth,DayOfWeek,UniqueCarrier,Origin,Dest,Distance,flight,begin_of_month,midddle_of_month,...,airport_origin_per_month,airport_dest_count,airport_origin_count,carrier_count,carrier_count_per month,deptime_cos,deptime_sin,flightUC,DestUC,OriginUC
0,8,21,7,1,19,82,732,171,0,0,...,1016,8290,11375,18024,1569,0.343660,-0.939094,265,494,67
1,4,20,3,19,226,180,834,3986,0,0,...,105,3523,1390,13069,1094,-0.612907,-0.790155,6907,1085,1441
2,9,2,5,21,239,62,416,4091,1,0,...,136,2246,1747,11737,977,-0.835807,-0.549023,7064,359,1518
3,11,25,6,16,81,184,872,1304,0,0,...,514,1785,6222,15343,1242,-0.884988,0.465615,2258,1122,484
4,10,7,6,20,182,210,423,2979,1,0,...,226,687,2571,30958,2674,0.073238,-0.997314,5144,1313,1103


Split the full_df back into training and testing sets based on the original training set size. 

In [ ]:
# Select the first N rows of full_df, where N is the number of rows in the original training set (train_df.shape[0]). These rows correspond to the training data.
X_train = full_df[:train_df.shape[0]]

# Select all rows from position N onward, which correspond to the test data.
X_test = full_df[train_df.shape[0]:]

Create a list of the categorical features.

In [20]:
categorical_features = ['Month',  'DayOfWeek', 'UniqueCarrier', 'Origin', 'Dest','flight',  'flightUC', 'DestUC', 'OriginUC']

Let's build a light GBM model to test the bayesian optimizer.

### [LightGBM](https://lightgbm.readthedocs.io/en/latest/) is a gradient boosting framework that uses tree-based learning algorithms. It is designed to be distributed and efficient with the following advantages:

* Faster training speed and higher efficiency.
* Lower memory usage.
* Better accuracy.
* Support of parallel and GPU learning.
* Capable of handling large-scale data.

First, we define the function we want to maximize and that will count cross-validation metrics of lightGBM for our parameters.

Some params such as num_leaves, max_depth, min_child_samples, min_data_in_leaf should be integers.

In [21]:
def lgb_eval(num_leaves,max_depth,lambda_l2,lambda_l1,min_child_samples, min_data_in_leaf):
    params = {
        "objective" : "binary",
        "metric" : "auc", 
        'is_unbalance': True,
        "num_leaves" : int(num_leaves),
        "max_depth" : int(max_depth),
        "lambda_l2" : lambda_l2,
        "lambda_l1" : lambda_l1,
        "num_threads" : 20,
        "min_child_samples" : int(min_child_samples),
        'min_data_in_leaf': int(min_data_in_leaf),
        "learning_rate" : 0.03,
        "subsample_freq" : 5,
        "bagging_seed" : 42,
        "verbosity" : -1
    }
    lgtrain = lightgbm.Dataset(X_train, y_train,categorical_feature=categorical_features)
    cv_result = lightgbm.cv(params,
                       lgtrain,
                       1000,
                       stratified=True,
                       nfold=3)
    return cv_result['valid auc-mean'][-1]

Apply the Bayesian optimizer to the function we created in the previous step to identify the best hyperparameters. We will run 5 iterations and set init_points = 2.


In [ ]:
# This instantiates the BayesianOptimization object and sets up the optimization problem. It defines the hyperparameter search space for LightGBM and specifies the evaluation function (lgb_eval) to maximize. It does not run the optimization yet.
lgbBO = BayesianOptimization(lgb_eval, {'num_leaves': (25, 4000),
                                                'max_depth': (5, 63),
                                                'lambda_l2': (0.0, 0.05),
                                                'lambda_l1': (0.0, 0.05),
                                                'min_child_samples': (50, 10000),
                                                'min_data_in_leaf': (100, 2000)
                                                })

# Run the Bayesian optimization process to find the best hyperparameters for LightGBM. It will perform 2 initial random evaluations followed by 5 iterations of optimization based on the results of previous evaluations.
lgbBO.maximize(n_iter=5, init_points=2)

|   iter    |  target   | num_le... | max_depth | lambda_l2 | lambda_l1 | min_ch... | min_da... |
-------------------------------------------------------------------------------------------------
| 1         | 0.7077317 | 2575.1339 | 29.267349 | 0.0479505 | 0.0016617 | 8209.8260 | 481.98282 |
| 2         | 0.7446203 | 3165.2643 | 7.8145650 | 0.0158542 | 0.0490501 | 8621.0647 | 1533.6027 |
| 3         | 0.7045613 | 2539.5028 | 29.420567 | 2.729e-05 | 0.0486207 | 8973.9556 | 399.32524 |
| 4         | 0.7444442 | 3384.1270 | 34.981534 | 0.0481228 | 0.0082995 | 8638.1601 | 1996.1340 |
| 5         | 0.7443930 | 3916.2592 | 22.448081 | 0.0010929 | 0.0044124 | 7363.9932 | 1874.1114 |
| 6         | 0.7399317 | 2405.8525 | 31.079910 | 0.0486732 | 0.0281269 | 7431.1260 | 1074.9293 |
| 7         | 0.7442239 | 2532.4620 | 14.433397 | 0.0104149 | 0.0065039 | 6400.3842 | 1912.2897 |


 **<font color='teal'> Print the best result by using the '.max' function.</font>**

In [23]:
lgbBO.max

{'target': np.float64(0.7446203654342098),
 'params': {'num_leaves': np.float64(3165.264336774781),
  'max_depth': np.float64(7.814565023929688),
  'lambda_l2': np.float64(0.01585423396187045),
  'lambda_l1': np.float64(0.04905013241269718),
  'min_child_samples': np.float64(8621.064737893403),
  'min_data_in_leaf': np.float64(1533.6027840178501)}}

Review the process at each step by using the '.res[0]' function.

In [24]:
lgbBO.res[0]

{'target': np.float64(0.7077317554865857),
 'params': {'num_leaves': np.float64(2575.133954871805),
  'max_depth': np.float64(29.267349993021227),
  'lambda_l2': np.float64(0.04795056154662234),
  'lambda_l1': np.float64(0.0016617914027494574),
  'min_child_samples': np.float64(8209.826014318536),
  'min_data_in_leaf': np.float64(481.9828295828724)}}